# Experiment 8 - CNN Regularization and Dropout

Compare a base CNN, L1, L2, dropout and combined L2 plus dropout.

**Run cells from top to bottom.** Installation is included where required.

In [ ]:
!pip -q install tensorflow matplotlib numpy scikit-learn

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.regularizers import l1, l2

X_train = X_train.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

def make_model(mode):
    dense = Dense(128, activation="relu")
    if mode == "L1":
        dense = Dense(128, activation="relu", kernel_regularizer=l1(0.001))
    elif mode == "L2":
        dense = Dense(128, activation="relu", kernel_regularizer=l2(0.001))
    layers = [Conv2D(32,(3,3),activation="relu",input_shape=(28,28,1)),
              MaxPooling2D((2,2)), Flatten(), dense]
    if mode in ["Dropout","L2+Dropout"]:
        if mode == "L2+Dropout":
            layers[-1] = Dense(128, activation="relu", kernel_regularizer=l2(0.001))
        layers.append(Dropout(0.5))
    return Sequential(layers + [Dense(10,activation="softmax")])

for mode in ["Base","L1","L2","Dropout","L2+Dropout"]:
    model = make_model(mode)
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    model.fit(X_train,y_train,epochs=1,batch_size=128,verbose=0)
    loss,acc = model.evaluate(X_test,y_test,verbose=0)
    print(mode, "test accuracy:", round(float(acc),4))